# Metabolomics VIP analyses

Date created: 12/22/2024

Notebook author: Yang Chen

Data analysis by: Britta De Pessemier

This notebook plots the following:
- Top VIPs separating between pairwise skin groups and univariate statistical testing

In [185]:
# Import Python packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.ticker as ticker
from matplotlib.patches import Rectangle
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import Normalize


In [186]:
VIP_LvsH_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_LvsH_merged.csv')
VIP_LvsH_merged['group'] = "H vs L"

VIP_NLvsH_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_NLvsH_merged.csv')
VIP_NLvsH_merged['group'] = "H vs NL"

VIP_NLvsL_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_NLvsL_merged.csv')
VIP_NLvsL_merged['group'] = "NL vs L"

VIP_LvsH = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_LvsH_Load.csv')
VIP_LvsH['group'] = "L vs H"

VIP_NLvsH = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_NLvsH_Load.csv')
VIP_NLvsH['group'] = "NL vs H"

VIP_NLvsL = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_NLvsL_Load.csv')
VIP_NLvsL['group'] = "NL vs L"
# Replace values in the GroupContrib column
VIP_NLvsL['GroupContrib'] = VIP_NLvsL['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Acne Non-lesional': 'Acne_NL'
})


In [206]:
# Concatenate the DataFrames vertically
VIP_combined = pd.concat([VIP_LvsH_merged, VIP_NLvsH_merged, VIP_NLvsL_merged], ignore_index=True)
VIP_combined_Loadings = pd.concat([VIP_LvsH, VIP_NLvsH, VIP_NLvsL], ignore_index=True)

# Sort the combined DataFrame by the 'VIP' column in descending order
VIP_sorted = VIP_combined.sort_values(by='VIP', ascending=False)

# Replace NaN values in 'superclass', 'superclass', and 'superclass' columns with "Unknown"
VIP_sorted[['superclass', 'class', 'subclass']] = VIP_sorted[['superclass', 'class', 'subclass']].fillna("Unknown")

VIP_sorted.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_all.csv')

# Filter out rows where VIP < 0.8
VIP_filtered = VIP_sorted[VIP_sorted['VIP'] >= 0.8]

# Filter out rows where 'molecular_formula' is NaN
VIP_filtered = VIP_filtered[VIP_filtered['molecular_formula'].notna()]

# Map the 'GroupContrib' column from VIP_combined_Loadings to VIP_filtered
VIP_filtered['GroupContrib'] = VIP_filtered['Feature'].map(
    dict(zip(VIP_combined_Loadings['ID'], VIP_combined_Loadings['GroupContrib']))
)

# Define a function to calculate the VIP_direction
def calculate_vip_direction(row):
    if row['group'] == 'L vs H':
        return row['VIP'] if row['GroupContrib'] == 'Acne_L' else -row['VIP']
    elif row['group'] == 'NL vs H':
        return row['VIP'] if row['GroupContrib'] == 'Acne_NL' else -row['VIP']
    elif row['group'] == 'NL vs L':
        return row['VIP'] if row['GroupContrib'] == 'Acne_L' else -row['VIP']
    else:
        return row['VIP']  # Default to keeping the value unchanged if no match

# Apply the function to create the new column
VIP_filtered['VIP_direction'] = VIP_filtered.apply(calculate_vip_direction, axis=1)

VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final.csv')

In [188]:
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_name.csv')

# Filter out rows where GroupContrib is NaN
VIP_filtered = VIP_filtered.dropna(subset=['GroupContrib'])
VIP_filtered.head()

,Unnamed: 0,Feature,VIP,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,MZErrorPPM,...,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,GroupContrib,VIP_direction,Shortened_Compound_Name
0,98,5930,2.460781,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,0.520785,...,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,Acne_NL,-2.460781,D-TRYPTOPHAN
1,0,29059,2.287412,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,1.960920,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,Acne_NL,2.287412,C19H36O4 Fatty acid
2,47,29059,2.215677,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,1.960920,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,Acne_NL,2.215677,C19H36O4 Fatty acid
3,100,24392,1.941285,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,6.554190,...,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,Acne_L,1.941285,Sorbitane Monooleate
4,49,5930,1.939163,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,0.520785,...,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,Acne_NL,1.939163,D-TRYPTOPHAN


In [189]:
color_map = {
    'Fatty Acyls': '#F28C28',  # Dark orange
    'Carboxylic acids and derivatives': '#4682B4',  # Steel blue
    'Indoles and derivatives': '#6A5ACD',  # Slate blue
    'Diarylheptanoids': '#66CDAA',  # Medium aquamarine
    'Organonitrogen compounds': '#C71585',  # Medium violet-red
    'Pyridines and derivatives': '#DAA520',  # Goldenrod
    'Azoles': '#8B0000',  # Dark red
    'Flavonoids': '#228B22',  # Forest green
    'Prenol lipids': '#CD5C5C',  # Indian red
    'Benzimidazoles': '#5F9EA0'  # Cadet blue
}

# Define unique groups in the dataset
unique_groups = ["H vs L", "H vs NL", "NL vs L"]  # Explicitly set the order of groups

# Create a 1x3 subplot with gridspec_kw for spacing
fig, axes = plt.subplots(
    1, 3, figsize=(18, 6), sharey=False,
    gridspec_kw={"wspace": 0.65}  # Adjust wspace as needed
)

# Add an overall title to the figure
fig.suptitle('Top Driving Metabolites Across Groups', fontsize=20, y=1.025)


# Define custom titles for each group
custom_titles = {
    "H vs L": "Healthy (left) vs Lesional (right)",
    "H vs NL": "Healthy (left) vs Non-lesional (right)",
    "NL vs L": "Non-lesional (left) vs Lesional (right)"
}

# Loop through each group and plot in the corresponding subplot
for i, group in enumerate(unique_groups):
    # Filter data for the current group
    group_data = VIP_filtered[VIP_filtered['group'] == group]
    vip_scores = group_data['VIP_direction']
    metabolites = group_data['Shortened_Compound_Name']
    classes = group_data['class']
    
    # Map colors for the current group
    dot_colors = classes.map(color_map).fillna('#808080')

    # Format y-tick labels to add a newline after each word
    formatted_metabolites = [label.replace(' ', '\n') for label in metabolites]

    # Plot data on the corresponding subplot
    scatter = axes[i].scatter(
        vip_scores, 
        range(len(group_data)), 
        color=dot_colors, 
        s=100  # Adjust the size of the dots (default is 20, increase as needed)
    )
    axes[i].axvline(x=1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=1
    axes[i].axvline(x=-1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=-1

    # Set y-ticks and y-tick labels for all subplots
    axes[i].set_yticks(range(len(metabolites)))
    axes[i].set_yticklabels(formatted_metabolites, fontsize=12)
    axes[i].tick_params(axis='y', labelleft=True)
    
    axes[i].invert_yaxis()
    axes[i].set_xlabel('VIP Scores', fontsize=12)
    
    # Set custom title for the subplot
    axes[i].set_title(custom_titles.get(group, f"({group})"), fontsize = 16, pad=10)

# Add a single legend at the bottom
legend_handles = [plt.Line2D([0], [0], marker='o', color=color, linestyle='', markersize=10, label=class_)
                  for class_, color in color_map.items()]
fig.legend(handles=legend_handles, title='Class', loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=12, title_fontsize=14, borderaxespad=-1)

# Adjust layout and save the figure
plt.tight_layout(rect=[0, 0.1, 1, 1])  # Adjust layout to leave space for the legend
output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot.png"
plt.savefig(output_path, dpi=600, bbox_inches='tight')
output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot.svg"
plt.savefig(output_path, bbox_inches='tight')
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/4079906049.py:75: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.1, 1, 1])  # Adjust layout to leave space for the legend
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/4079906049.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Perform univariate statistical testing on above metabolites w/ top VIPs

In [190]:
# Read in metabolomics table
metaB_table = pd.read_csv('../Data/metabolomics/Run3_10252024/data_sample_10282024.csv')

# Set the 'SampleID' column as the index
metaB_table.set_index('SampleID', inplace=True)

# Convert columns in metaB_table to int (if applicable)
metaB_table.columns = metaB_table.columns.astype(int)

# Convert features_to_keep to int
features_to_keep = VIP_filtered['Feature'].unique().astype(int)

# Subset metaB_table using the integer feature list
metaB_table_subset = metaB_table.loc[:, metaB_table.columns.isin(features_to_keep)]

metaB_table_subset

,655,777,2969,5930,3794,29059,30129,15963,27758,23297,24392
SampleID,,,,,,,,,,,
LAMI.RD001.D0.C1,39859.7580,0.00,970487.500,567510.060,12406.8260,611709.700,25936.0230,0.0000,16778.838,0.0000,0.00
LAMI.RD306.D9.C2,0.0000,178911.47,1071868.200,672054.400,20433.4900,437938.200,23435.4340,6174.2900,12865.390,0.0000,0.00
LAMI.RD308.D2.C2,3065.1714,0.00,750870.100,442653.660,5348.9400,231525.840,0.0000,21603.1600,14299.352,0.0000,0.00
LAMI.RD304.D11.C1,36750.4000,0.00,595568.940,328151.120,9403.6520,78491.530,1793.8712,15929.3220,0.000,5375.8525,99470.69
LAMI.RD304.D11.C2,0.0000,0.00,303845.300,152392.720,2286.2397,261996.800,8281.6990,28877.0040,15780.309,2691.0256,157503.92
...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,0.00,579058.800,498893.780,5144.6616,122958.260,9814.8600,70822.4450,56654.043,0.0000,0.00
LAMI.RD308.D9.C3,3076.9688,0.00,281896.700,262886.470,0.0000,65659.700,5371.9844,5548.9746,21826.117,0.0000,0.00
LAMI.RD319.D2.C2,0.0000,0.00,255542.720,121803.890,0.0000,73231.560,1726.0502,17378.8120,19741.246,0.0000,0.00


In [191]:
# Read in metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')

# Subset metadata based on the indices of metaB_subset
metadata_subset = metadata[metadata['SampleID'].isin(metaB_table_subset.index)]

metadata_subset

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
6,LAMI.RD317.D14.C2,C2,70,Lesional,skin,Day 14,44,14,317,15,...,44,70,25.042556,acne,RD317,acne,PP_317,PP_317C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [192]:
# Map the 'group' column from metadata_subset to metaB_subset
metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])

# Replace values in the 'group' column
metaB_table_subset['group'] = metaB_table_subset['group'].replace({
    'Healthy': 'H',
    'Acne_L': 'L',
    'Acne_NL': 'NL'
})

metaB_table_subset

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/2462488200.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/2462488200.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset['group'].replace({


,655,777,2969,5930,3794,29059,30129,15963,27758,23297,24392,group
SampleID,,,,,,,,,,,,
LAMI.RD001.D0.C1,39859.7580,0.00,970487.500,567510.060,12406.8260,611709.700,25936.0230,0.0000,16778.838,0.0000,0.00,H
LAMI.RD306.D9.C2,0.0000,178911.47,1071868.200,672054.400,20433.4900,437938.200,23435.4340,6174.2900,12865.390,0.0000,0.00,L
LAMI.RD308.D2.C2,3065.1714,0.00,750870.100,442653.660,5348.9400,231525.840,0.0000,21603.1600,14299.352,0.0000,0.00,L
LAMI.RD304.D11.C1,36750.4000,0.00,595568.940,328151.120,9403.6520,78491.530,1793.8712,15929.3220,0.000,5375.8525,99470.69,L
LAMI.RD304.D11.C2,0.0000,0.00,303845.300,152392.720,2286.2397,261996.800,8281.6990,28877.0040,15780.309,2691.0256,157503.92,L
...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,0.00,579058.800,498893.780,5144.6616,122958.260,9814.8600,70822.4450,56654.043,0.0000,0.00,NL
LAMI.RD308.D9.C3,3076.9688,0.00,281896.700,262886.470,0.0000,65659.700,5371.9844,5548.9746,21826.117,0.0000,0.00,NL
LAMI.RD319.D2.C2,0.0000,0.00,255542.720,121803.890,0.0000,73231.560,1726.0502,17378.8120,19741.246,0.0000,0.00,L


In [193]:
VIP_filtered['Feature'].unique()

array([ 5930, 29059, 24392,  3794,  2969, 30129, 27758,   777,   655,
       15963, 23297])

In [194]:
# Prepare the dictionary to set up univariate testing
feature_group_dict = (
    VIP_filtered.groupby('Feature')['group']
    .apply(lambda x: x.unique().tolist())
    .to_dict()
)

feature_group_dict

{655: ['NL vs L', 'H vs L'],
 777: ['H vs NL', 'NL vs L'],
 2969: ['H vs NL', 'NL vs L', 'H vs L'],
 3794: ['H vs L', 'NL vs L'],
 5930: ['NL vs L', 'H vs NL', 'H vs L'],
 15963: ['H vs NL', 'NL vs L'],
 23297: ['H vs L', 'NL vs L'],
 24392: ['NL vs L', 'H vs NL'],
 27758: ['H vs L', 'H vs NL'],
 29059: ['H vs L', 'H vs NL', 'NL vs L'],
 30129: ['H vs NL', 'H vs L']}

In [195]:
# Initialize an empty dictionary to store results
statistical_results = {}

# Iterate through each metabolite in feature_group_dict
for metabolite, comparisons in feature_group_dict.items():
    # Create a dictionary to store results for the current metabolite
    metabolite_results = {}
    
    # Get the values for the metabolite from metaB_table_subset
    metabolite_values = metaB_table_subset[metabolite]
    
    # Iterate through each comparison for the current metabolite
    for comparison in comparisons:
        # Split the comparison into X and Y
        group_x, group_y = comparison.split(" vs ")
        
        # Subset values for the two groups
        values_x = metabolite_values[metaB_table_subset['group'] == group_x]
        values_y = metabolite_values[metaB_table_subset['group'] == group_y]
        
        # Perform the Mann-Whitney U test
        stat, p_value = mannwhitneyu(values_x, values_y, alternative='two-sided')
        
        # Store the result
        metabolite_results[comparison] = {'U-statistic': stat, 'p-value': p_value}
    
    # Add results for the current metabolite to the main results dictionary
    statistical_results[metabolite] = metabolite_results

# Convert results to a DataFrame for easier viewing (optional)
results_df = pd.DataFrame.from_dict({(met, comp): vals 
                                     for met, comps in statistical_results.items() 
                                     for comp, vals in comps.items()}, 
                                    orient='index')


# View results
results_df


U-statistic   p-value
655   NL vs L       3590.0  0.041844
      H vs L        3391.0  0.850968
777   H vs NL       1415.0  0.007137
      NL vs L       2953.0  0.888380
2969  H vs NL        772.0  0.018734
      NL vs L       3543.0  0.066897
      H vs L        2776.0  0.083912
3794  H vs L        3029.5  0.343141
      NL vs L       3638.0  0.030930
5930  NL vs L       3785.0  0.008343
      H vs NL        721.0  0.006084
      H vs L        2737.0  0.064346
15963 H vs NL       1057.5  0.877484
      NL vs L       3183.0  0.524495
23297 H vs L        2371.0  0.000082
      NL vs L       3284.0  0.249879
24392 NL vs L       2417.0  0.043213
      H vs NL       1068.0  0.934584
27758 H vs L        2838.5  0.125025
      H vs NL        837.5  0.064745
29059 H vs L        2601.0  0.023030
      H vs NL        706.0  0.004254
      NL vs L       3467.0  0.114245
30129 H vs NL        720.0  0.005618
      H vs L        2655.0  0.034444

In [196]:
# Extract p-values from the results DataFrame
results_df = pd.DataFrame.from_dict(
    {
        (met, comp): vals
        for met, comps in statistical_results.items()
        for comp, vals in comps.items()
    },
    orient="index"
)

# Apply Benjamini-Hochberg p-value correction
results_df['p_value_bh'] = multipletests(results_df['p-value'], method='fdr_bh')[1]

# View updated DataFrame with corrected p-values
results_df


U-statistic   p-value  p_value_bh
655   NL vs L       3590.0  0.041844    0.090027
      H vs L        3391.0  0.850968    0.925395
777   H vs NL       1415.0  0.007137    0.034761
      NL vs L       2953.0  0.888380    0.925395
2969  H vs NL        772.0  0.018734    0.066906
      NL vs L       3543.0  0.066897    0.111494
      H vs L        2776.0  0.083912    0.131112
3794  H vs L        3029.5  0.343141    0.428926
      NL vs L       3638.0  0.030930    0.085918
5930  NL vs L       3785.0  0.008343    0.034761
      H vs NL        721.0  0.006084    0.034761
      H vs L        2737.0  0.064346    0.111494
15963 H vs NL       1057.5  0.877484    0.925395
      NL vs L       3183.0  0.524495    0.624399
23297 H vs L        2371.0  0.000082    0.002051
      NL vs L       3284.0  0.249879    0.328788
24392 NL vs L       2417.0  0.043213    0.090027
      H vs NL       1068.0  0.934584    0.934584
27758 H vs L        2838.5  0.125025    0.173646
      H vs NL        837.5  0.064745    0.111494
29059 H vs L        2601.0  0.023030    0.071969
      H vs NL        706.0  0.004254    0.034761
      NL vs L       3467.0  0.114245    0.168007
30129 H vs NL        720.0  0.005618    0.034761
      H vs L        2655.0  0.034444    0.086109

In [197]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='Feature')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['Feature'] = VIP_filtered_unique['Feature'].astype(int)

# Create a mapping from Feature to Shortened_Compound_Name
feature_to_name_mapping = VIP_filtered_unique.set_index('Feature')['Shortened_Compound_Name']

# Map the Shortened_Compound_Name to results_df
results_df['Shortened_Compound_Name'] = feature_index.map(feature_to_name_mapping)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/2105925968.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['Feature'] = VIP_filtered_unique['Feature'].astype(int)


In [198]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='Feature')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['Feature'] = VIP_filtered_unique['Feature'].astype(int)

# Create a mapping from Feature to VIP values
feature_to_vip_mapping = VIP_filtered_unique.set_index('Feature')['VIP']

# Map the VIP values to results_df
results_df['VIP'] = feature_index.map(feature_to_vip_mapping)

# Sort results_df by the 'VIP' column in descending order
results_df = results_df.sort_values(by='VIP', ascending=False)

results_df

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/508360414.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['Feature'] = VIP_filtered_unique['Feature'].astype(int)


U-statistic   p-value  p_value_bh  \
5930  NL vs L       3785.0  0.008343    0.034761   
      H vs NL        721.0  0.006084    0.034761   
      H vs L        2737.0  0.064346    0.111494   
29059 NL vs L       3467.0  0.114245    0.168007   
      H vs L        2601.0  0.023030    0.071969   
      H vs NL        706.0  0.004254    0.034761   
24392 H vs NL       1068.0  0.934584    0.934584   
      NL vs L       2417.0  0.043213    0.090027   
3794  H vs L        3029.5  0.343141    0.428926   
      NL vs L       3638.0  0.030930    0.085918   
2969  H vs NL        772.0  0.018734    0.066906   
      NL vs L       3543.0  0.066897    0.111494   
      H vs L        2776.0  0.083912    0.131112   
30129 H vs L        2655.0  0.034444    0.086109   
      H vs NL        720.0  0.005618    0.034761   
27758 H vs NL        837.5  0.064745    0.111494   
      H vs L        2838.5  0.125025    0.173646   
777   NL vs L       2953.0  0.888380    0.925395   
      H vs NL       1415.0  0.007137    0.034761   
655   NL vs L       3590.0  0.041844    0.090027   
      H vs L        3391.0  0.850968    0.925395   
15963 NL vs L       3183.0  0.524495    0.624399   
      H vs NL       1057.5  0.877484    0.925395   
23297 NL vs L       3284.0  0.249879    0.328788   
      H vs L        2371.0  0.000082    0.002051   

                        Shortened_Compound_Name       VIP  
5930  NL vs L                      D-TRYPTOPHAN  2.460781  
      H vs NL                      D-TRYPTOPHAN  2.460781  
      H vs L                       D-TRYPTOPHAN  2.460781  
29059 NL vs L               C19H36O4 Fatty acid  2.287412  
      H vs L                C19H36O4 Fatty acid  2.287412  
      H vs NL               C19H36O4 Fatty acid  2.287412  
24392 H vs NL              Sorbitane Monooleate  1.941285  
      NL vs L              Sorbitane Monooleate  1.941285  
3794  H vs L                   Glutamyltyrosine  1.779836  
      NL vs L                  Glutamyltyrosine  1.779836  
2969  H vs NL                     Phenylalanine  1.686018  
      NL vs L                     Phenylalanine  1.686018  
      H vs L                      Phenylalanine  1.686018  
30129 H vs L                C19H38O4 Fatty acid  1.522016  
      H vs NL               C19H38O4 Fatty acid  1.522016  
27758 H vs NL              N-Oleoylethanolamine  1.491353  
      H vs L               N-Oleoylethanolamine  1.491353  
777   NL vs L                          Nicotine  1.281435  
      H vs NL                          Nicotine  1.281435  
655   NL vs L                     Urocanic acid  1.132097  
      H vs L                      Urocanic acid  1.132097  
15963 NL vs L  C19H22O4 Linear diarylheptanoids  0.952309  
      H vs NL  C19H22O4 Linear diarylheptanoids  0.952309  
23297 NL vs L                         Gln-C14:0  0.950006  
      H vs L                          Gln-C14:0  0.950006

In [199]:
# Filter rows where the p-value is greater than 0.1
filtered_features = results_df[results_df['p-value'] <= 0.1]
filtered_features

U-statistic   p-value  p_value_bh Shortened_Compound_Name  \
5930  NL vs L       3785.0  0.008343    0.034761            D-TRYPTOPHAN   
      H vs NL        721.0  0.006084    0.034761            D-TRYPTOPHAN   
      H vs L        2737.0  0.064346    0.111494            D-TRYPTOPHAN   
29059 H vs L        2601.0  0.023030    0.071969     C19H36O4 Fatty acid   
      H vs NL        706.0  0.004254    0.034761     C19H36O4 Fatty acid   
24392 NL vs L       2417.0  0.043213    0.090027    Sorbitane Monooleate   
3794  NL vs L       3638.0  0.030930    0.085918        Glutamyltyrosine   
2969  H vs NL        772.0  0.018734    0.066906           Phenylalanine   
      NL vs L       3543.0  0.066897    0.111494           Phenylalanine   
      H vs L        2776.0  0.083912    0.131112           Phenylalanine   
30129 H vs L        2655.0  0.034444    0.086109     C19H38O4 Fatty acid   
      H vs NL        720.0  0.005618    0.034761     C19H38O4 Fatty acid   
27758 H vs NL        837.5  0.064745    0.111494    N-Oleoylethanolamine   
777   H vs NL       1415.0  0.007137    0.034761                Nicotine   
655   NL vs L       3590.0  0.041844    0.090027           Urocanic acid   
23297 H vs L        2371.0  0.000082    0.002051               Gln-C14:0   

                    VIP  
5930  NL vs L  2.460781  
      H vs NL  2.460781  
      H vs L   2.460781  
29059 H vs L   2.287412  
      H vs NL  2.287412  
24392 NL vs L  1.941285  
3794  NL vs L  1.779836  
2969  H vs NL  1.686018  
      NL vs L  1.686018  
      H vs L   1.686018  
30129 H vs L   1.522016  
      H vs NL  1.522016  
27758 H vs NL  1.491353  
777   H vs NL  1.281435  
655   NL vs L  1.132097  
23297 H vs L   0.950006

In [200]:
# Group by the primary index and collect the secondary index values
keep_dict = filtered_features.groupby(level=0).apply(lambda df: df.index.get_level_values(1).tolist()).to_dict()

keep_dict

{655: ['NL vs L'],
 777: ['H vs NL'],
 2969: ['H vs NL', 'NL vs L', 'H vs L'],
 3794: ['NL vs L'],
 5930: ['NL vs L', 'H vs NL', 'H vs L'],
 23297: ['H vs L'],
 24392: ['NL vs L'],
 27758: ['H vs NL'],
 29059: ['H vs L', 'H vs NL'],
 30129: ['H vs L', 'H vs NL']}

In [201]:
# Function to check if the Feature and group combination is in keep_dict
def filter_rows(row):
    feature = row['Feature']
    group = row['group']
    return feature in keep_dict and group in keep_dict[feature]

# Apply the filter function to VIP_filtered
VIP_filtered_subset = VIP_filtered[VIP_filtered.apply(filter_rows, axis=1)]

VIP_filtered_subset

,Unnamed: 0,Feature,VIP,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,MZErrorPPM,...,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,GroupContrib,VIP_direction,Shortened_Compound_Name
0,98,5930,2.460781,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,0.520785,...,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,Acne_NL,-2.460781,D-TRYPTOPHAN
1,0,29059,2.287412,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,1.960920,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,Acne_NL,2.287412,C19H36O4 Fatty acid
2,47,29059,2.215677,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,1.960920,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,Acne_NL,2.215677,C19H36O4 Fatty acid
3,100,24392,1.941285,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,6.554190,...,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,Acne_L,1.941285,Sorbitane Monooleate
4,49,5930,1.939163,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,0.520785,...,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,Acne_NL,1.939163,D-TRYPTOPHAN
6,51,2969,1.686018,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,0,0.643110,...,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,H vs NL,Acne_NL,1.686018,Phenylalanine
7,102,2969,1.622320,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,0,0.643110,...,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,NL vs L,Acne_NL,-1.622320,Phenylalanine
8,52,30129,1.522016,CCMSLIB00005723208,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.608127,163100,0,1.266400,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005723208,H vs NL,Healthy,1.522016,C19H38O4 Fatty acid
10,53,27758,1.461870,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,282600,0,0.374099,...,Organonitrogen compounds,Amines,Fatty amides,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,Acne_NL,1.461870,N-Oleoylethanolamine
12,8,30129,1.379850,CCMSLIB00005723208,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.608127,163100,0,1.266400,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005723208,H vs L,Healthy,1.379850,C19H38O4 Fatty acid


In [202]:
# Ensure Feature in VIP_filtered_subset and the primary index in filtered_features are the same type
VIP_filtered_subset['Feature'] = VIP_filtered_subset['Feature'].astype(int)
filtered_features.index = filtered_features.index.set_levels(
    filtered_features.index.levels[0].astype(int), level=0
)

# Create a MultiIndex mapping of (Feature, group) to p-value
p_value_mapping = filtered_features['p-value']

# Create a new column in VIP_filtered_subset for the p_value_bh
VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['Feature', 'group']).index.map(p_value_mapping)

# Reset the index to return to the original structure if needed
VIP_filtered_subset.reset_index(drop=True, inplace=True)

# View the updated DataFrame
VIP_filtered_subset


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/2959964211.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['Feature'] = VIP_filtered_subset['Feature'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/2959964211.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['Feature', 'group']).index.map(p_value_mapping)


,Unnamed: 0,Feature,VIP,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,MZErrorPPM,...,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,GroupContrib,VIP_direction,Shortened_Compound_Name,p-value
0,98,5930,2.460781,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,0.520785,...,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,Acne_NL,-2.460781,D-TRYPTOPHAN,0.008343
1,0,29059,2.287412,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,1.960920,...,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,Acne_NL,2.287412,C19H36O4 Fatty acid,0.023030
2,47,29059,2.215677,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,1.960920,...,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,Acne_NL,2.215677,C19H36O4 Fatty acid,0.004254
3,100,24392,1.941285,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,6.554190,...,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,Acne_L,1.941285,Sorbitane Monooleate,0.043213
4,49,5930,1.939163,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,0.520785,...,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,Acne_NL,1.939163,D-TRYPTOPHAN,0.006084
5,51,2969,1.686018,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,0,0.643110,...,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,H vs NL,Acne_NL,1.686018,Phenylalanine,0.018734
6,102,2969,1.622320,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,0,0.643110,...,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,NL vs L,Acne_NL,-1.622320,Phenylalanine,0.066897
7,52,30129,1.522016,CCMSLIB00005723208,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.608127,163100,0,1.266400,...,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005723208,H vs NL,Healthy,1.522016,C19H38O4 Fatty acid,0.005618
8,53,27758,1.461870,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,282600,0,0.374099,...,Amines,Fatty amides,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,Acne_NL,1.461870,N-Oleoylethanolamine,0.064745
9,8,30129,1.379850,CCMSLIB00005723208,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.608127,163100,0,1.266400,...,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005723208,H vs L,Healthy,1.379850,C19H38O4 Fatty acid,0.034444


In [203]:
# Create a 1x3 subplot with gridspec_kw for spacing
fig, axes = plt.subplots(
    1, 3, figsize=(22, 6), sharey=False,  # Increase width to make space for heatmaps
    gridspec_kw={"wspace": 0.75}  # Adjust wspace as needed
)

# Add an overall title to the figure
fig.suptitle('Top Driving Metabolites Across Groups with Statistical Significance Heatmaps', fontsize=20, y=1.025)

# Loop through each group and plot in the corresponding subplot
for i, group in enumerate(unique_groups):
    # Filter data for the current group
    group_data = VIP_filtered_subset[VIP_filtered_subset['group'] == group]
    vip_scores = group_data['VIP_direction']
    metabolites = group_data['Shortened_Compound_Name']
    classes = group_data['class']
    p_values = group_data['p-value']  # Use raw p-values directly

    # Map colors for the current group
    dot_colors = classes.map(color_map).fillna('#808080')

    # Format y-tick labels to add a newline after each word
    formatted_metabolites = [label.replace(' ', '\n') for label in metabolites]

    # Plot scatter plot on the left
    scatter = axes[i].scatter(
        vip_scores,
        range(len(group_data)),
        color=dot_colors,
        s=100  # Adjust the size of the dots
    )
    axes[i].axvline(x=1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=1
    axes[i].axvline(x=-1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=-1

    # Set y-ticks and y-tick labels for scatter plot
    axes[i].set_yticks(range(len(metabolites)))
    axes[i].set_yticklabels(formatted_metabolites, fontsize=12)
    axes[i].tick_params(axis='y', labelleft=True)
    axes[i].invert_yaxis()
    axes[i].set_xlabel('VIP Scores', fontsize=12)
    axes[i].set_title(custom_titles.get(group, f"({group})"), fontsize=16, pad=10)

    # Add heatmap to the right of each subplot
    ax_heatmap = fig.add_axes([axes[i].get_position().x1 + 0.02,  # Position heatmap to the right of scatter
                               axes[i].get_position().y0,
                               0.01,
                               axes[i].get_position().height])
    
    # Use raw p-values for visualization
    p_values_array = p_values.values.reshape(-1, 1)  # Convert to 2D array for heatmap

    # Generate tick positions for the colorbar
    p_value_min = VIP_filtered_subset['p-value'].min()
    p_value_max = VIP_filtered_subset['p-value'].max()

    # Normalize the color scale for the heatmap
    norm = Normalize(vmin=p_value_min, vmax=p_value_max)

    heatmap = ax_heatmap.imshow(
        p_values_array,
        norm=norm,
        cmap=sns.color_palette("Blues_r", as_cmap=True),  # Use a color map for the heatmap
        aspect="auto",
        origin="lower"
    )

    # Set heatmap ticks to match scatter plot
    ax_heatmap.set_yticks(range(len(metabolites)))
    ax_heatmap.set_yticklabels([])  # Hide y-tick labels for heatmap
    ax_heatmap.tick_params(axis='y', length=0)  # Remove tick marks
    ax_heatmap.set_xticks([])  # Hide x-tick labels
    ax_heatmap.set_title('p-value', fontsize=12)

# Add a single legend at the bottom
legend_handles = [plt.Line2D([0], [0], marker='o', color=color, linestyle='', markersize=12, label=class_)
                  for class_, color in color_map.items()]
fig.legend(handles=legend_handles, title='Class', loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=12, title_fontsize=14, labelspacing=1.5, borderaxespad=-3)

# Add a colorbar for the heatmaps
cbar_ax = fig.add_axes([0.35, -0.3, 0.3, 0.02])  # Position for colorbar at the bottom
cbar = plt.colorbar(heatmap, cax=cbar_ax, orientation='horizontal')

# Use np.linspace to generate evenly spaced ticks
tick_positions = np.linspace(p_value_min, p_value_max, num=5)
print(tick_positions)

# Format tick labels as plain decimal values
tick_labels = [f"{tick:.6f}" for tick in tick_positions]  # Adjust the number of decimal places as needed
print(tick_labels)

# Set colorbar ticks and labels
cbar.set_ticks(tick_positions)
cbar.set_ticklabels(tick_labels, fontsize=12)
cbar.set_label('p-value', fontsize=14, labelpad=20)
cbar.ax.xaxis.set_label_position('top')

# Adjust layout and save the figure
plt.tight_layout(rect=[0, 0.1, 0.9, 1])  # Adjust layout to leave space for the legend and colorbar
output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot_with_pvalue_heatmaps.png"
plt.savefig(output_path, dpi=600, bbox_inches='tight')
output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot_with_pvalue_heatmaps.svg"
plt.savefig(output_path, bbox_inches='tight')

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_63768/2862902724.py:98: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.1, 0.9, 1])  # Adjust layout to leave space for the legend and colorbar


[8.20593154e-05 2.10395264e-02 4.19969935e-02 6.29544607e-02
 8.39119278e-02]
['0.000082', '0.021040', '0.041997', '0.062954', '0.083912']


In [204]:
# Create a mapping from Feature to Shortened_Compound_Name
feature_to_name = dict(zip(VIP_filtered_unique['Feature'], VIP_filtered_unique['Shortened_Compound_Name']))

# Replace column names in metaB_table_subset using the mapping
metaB_table_subset_renamed = metaB_table_subset.rename(columns=feature_to_name)

# Save table as CSV for multi-omics analysis
metaB_table_subset_renamed.to_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs.csv')

In [205]:
# # Create a 1x3 subplot with gridspec_kw for spacing
# fig, axes = plt.subplots(
#     1, 3, figsize=(22, 6), sharey=False,  # Increase width to make space for heatmaps
#     gridspec_kw={"wspace": 0.75}  # Adjust wspace as needed
# )

# # Add an overall title to the figure
# fig.suptitle('Top Driving Metabolites Across Groups with Statistical Significance Heatmaps', fontsize=20, y=1.025)

# # Define max and min bh corrected p-values to inform scale of heatmap
# p_value_bh_min = 0.00205148
# p_value_bh_max = VIP_filtered_subset['p_value_bh'].max()

# # Loop through each group and plot in the corresponding subplot
# for i, group in enumerate(unique_groups):
#     # Filter data for the current group
#     group_data = VIP_filtered_subset[VIP_filtered_subset['group'] == group]
#     vip_scores = group_data['VIP_direction']
#     metabolites = group_data['Shortened_Compound_Name']
#     classes = group_data['class']
#     p_values = group_data['p_value_bh']  # Use raw p_value_bh directly
#     print(p_values)

#     # Map colors for the current group
#     dot_colors = classes.map(color_map).fillna('#808080')

#     # Format y-tick labels to add a newline after each word
#     formatted_metabolites = [label.replace(' ', '\n') for label in metabolites]

#     # Plot scatter plot on the left
#     scatter = axes[i].scatter(
#         vip_scores,
#         range(len(group_data)),
#         color=dot_colors,
#         s=100  # Adjust the size of the dots
#     )
#     axes[i].axvline(x=1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=1
#     axes[i].axvline(x=-1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=-1

#     # Set y-ticks and y-tick labels for scatter plot
#     axes[i].set_yticks(range(len(metabolites)))
#     axes[i].set_yticklabels(formatted_metabolites, fontsize=12)
#     axes[i].tick_params(axis='y', labelleft=True)
#     axes[i].invert_yaxis()
#     axes[i].set_xlabel('VIP Scores', fontsize=12)
#     axes[i].set_title(custom_titles.get(group, f"({group})"), fontsize=16, pad=10)

#     # Add heatmap to the right of each subplot
#     ax_heatmap = fig.add_axes([axes[i].get_position().x1 + 0.02,  # Position heatmap to the right of scatter
#                                axes[i].get_position().y0,
#                                0.01,
#                                axes[i].get_position().height])
    
#     # Use p_value_bh for visualization
#     p_values_array = p_values.values.reshape(-1, 1)  # Convert to 2D array for heatmap

#     heatmap = ax_heatmap.imshow(
#         p_values_array,
#         cmap=sns.color_palette("Blues_r", as_cmap=True),  # Use a color map for the heatmap
#         aspect="auto",
#         origin="lower",
#         vmin=p_value_bh_min,
#         vmax=p_value_bh_max  # Ensure the heatmap uses the full range
#     )

#     # Set heatmap ticks to match scatter plot
#     ax_heatmap.set_yticks(range(len(metabolites)))
#     ax_heatmap.set_yticklabels([])  # Hide y-tick labels for heatmap
#     ax_heatmap.tick_params(axis='y', length=0)  # Remove tick marks
#     ax_heatmap.set_xticks([])  # Hide x-tick labels
#     ax_heatmap.set_title('p_value_bh', fontsize=12)

# # Add a single legend at the bottom
# legend_handles = [plt.Line2D([0], [0], marker='o', color=color, linestyle='', markersize=12, label=class_)
#                   for class_, color in color_map.items()]
# fig.legend(handles=legend_handles, title='Class', loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=12, title_fontsize=14, labelspacing=1.5, borderaxespad=-3)


# # Generate tick positions that align with the colormap scale
# tick_positions = np.linspace(p_value_bh_min, p_value_bh_max, num=5)  # Use 5 evenly spaced ticks

# # Create tick labels that match these positions
# tick_labels = [f'{tick:.3f}' for tick in tick_positions]

# # Add a colorbar for the heatmaps
# cbar_ax = fig.add_axes([0.35, -0.3, 0.3, 0.02])  # Position for colorbar at the bottom
# cbar = plt.colorbar(heatmap, cax=cbar_ax, orientation='horizontal')

# # Set colorbar ticks and labels
# cbar.set_ticks(tick_positions)
# cbar.set_ticklabels(tick_labels)

# # Label the colorbar
# cbar.set_label('p_value_bh', fontsize=14, labelpad=15)
# cbar.ax.xaxis.set_label_position('top')

# # Adjust layout and save the figure
# plt.tight_layout(rect=[0, 0.1, 0.9, 1])  # Adjust layout to leave space for the legend and colorbar
# output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot_with_pvalue-bh_heatmaps.png"
# plt.savefig(output_path, dpi=600, bbox_inches='tight')
# output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot_with_pvalue-bh_heatmaps.svg"
# plt.savefig(output_path, bbox_inches='tight')
# plt.show()
